### **RSA: Key Generation, Textbook Pitfalls, and OAEP**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrafa1607/Cryptography_In_Python/blob/main/Algorithms/RSA_Cryptosystem.ipynb)

> **RSA** encrypts with `c = m^e mod n` and decrypts with `m = c^d mod n`, where `n = p · q` and `d` is the inverse of `e` modulo `λ(n)`. Its security depends on the difficulty of **factoring** `n`. This notebook builds RSA from scratch, then shows two classic failures: **textbook RSA applied character by character** (broken without the private key) and **small keys** (factored in seconds). It ends with the safe way: **OAEP** for encryption and **PSS** for signatures.

> [!WARNING]
> Educational code. Never use these implementations to protect real data: use a vetted library (for example, `cryptography`) with modern, authenticated algorithms.

In [1]:
# Key generation with Miller-Rabin, secure randomness, and e = 65537
import math
import secrets
import time

def is_probable_prime(n, rounds=40):
    if n < 2:
        return False
    for small in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29):
        if n % small == 0:
            return n == small
    d, s = n - 1, 0
    while d % 2 == 0:
        d, s = d // 2, s + 1
    for _ in range(rounds):
        x = pow(secrets.randbelow(n - 3) + 2, d, n)
        if x in (1, n - 1):
            continue
        for _ in range(s - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break
        else:
            return False
    return True

def random_prime(bits, e=65537):
    while True:
        p = secrets.randbits(bits) | (1 << (bits - 1)) | 1
        if is_probable_prime(p) and math.gcd(p - 1, e) == 1:
            return p

def generate_keys(bits=1024, e=65537):
    p, q = random_prime(bits // 2), random_prime(bits // 2)
    lam = math.lcm(p - 1, q - 1)                      # Carmichael function λ(n)
    return (e, p * q), (pow(e, -1, lam), p * q)       # public (e, n), private (d, n)

public, private = generate_keys(1024)
e, n = public
message = int.from_bytes(b"RSA from scratch", "big")
c = pow(message, e, n)
m = pow(c, private[0], n)
print(f"n has {n.bit_length()} bits | e = {e}")
print("Decrypted:", m.to_bytes((m.bit_length() + 7) // 8, "big").decode())
assert m == message

n has 1023 bits | e = 65537
Decrypted: RSA from scratch


#### **01. Pitfall: Textbook RSA, One Character at a Time**

Textbook RSA is **deterministic**: the same input always gives the same output. Encrypting each character separately turns RSA into a substitution cipher with only ~100 possible inputs, so anyone with the **public key** can encrypt every character and build a reverse lookup table. No factoring needed.

In [2]:
# Decrypt per-character textbook RSA using only the public key
secret_text = "Transfer 9,500 USD to account 4471 on Friday."
intercepted = [pow(ord(ch), e, n) for ch in secret_text]        # what the victim sends

lookup = {pow(code, e, n): chr(code) for code in range(32, 127)}  # attacker: public key only
recovered = "".join(lookup[c] for c in intercepted)
print("Recovered without the private key:", recovered)
assert recovered == secret_text

Recovered without the private key: Transfer 9,500 USD to account 4471 on Friday.


#### **02. Pitfall: Small Keys**

A 64-bit modulus falls to **Pollard's rho** factoring in a fraction of a second, which reveals `p`, `q`, and therefore the private key. RSA-768 was factored in 2009; current guidance is **at least 2048 bits**, with 3072 bits for use beyond 2030.

In [3]:
# Factor a 64-bit RSA modulus with Pollard's rho and rebuild the private key
def pollard_rho(n):
    if n % 2 == 0:
        return 2
    while True:
        x = y = secrets.randbelow(n - 2) + 2
        c, d = secrets.randbelow(n - 1) + 1, 1
        while d == 1:
            x = (x * x + c) % n
            y = (y * y + c) % n
            y = (y * y + c) % n
            d = math.gcd(abs(x - y), n)
        if d != n:
            return d

small_public, small_private = generate_keys(64)
e_small, n_small = small_public
start = time.perf_counter()
p = pollard_rho(n_small)
q = n_small // p
d_recovered = pow(e_small, -1, math.lcm(p - 1, q - 1))
print(f"n = {n_small} factored in {time.perf_counter() - start:.3f} s: {p} x {q}")
print("Private key recovered:", d_recovered == small_private[0])
assert d_recovered == small_private[0]

n = 8396237054623147853 factored in 0.036 s: 2508357763 x 3347304431
Private key recovered: True


#### **03. The Safe Way: OAEP and PSS**

* **OAEP** adds randomized padding: encrypting the same message twice gives different ciphertexts, and malformed ciphertexts are rejected.
* **PSS** is the randomized padding for signatures. A signature proves who signed and that the message was not changed.

In [4]:
# RSA-OAEP encryption and RSA-PSS signatures with the cryptography library
from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding, rsa

private_key = rsa.generate_private_key(public_exponent=65537, key_size=3072)
public_key = private_key.public_key()

oaep = padding.OAEP(mgf=padding.MGF1(hashes.SHA256()), algorithm=hashes.SHA256(), label=None)
c1 = public_key.encrypt(b"same message", oaep)
c2 = public_key.encrypt(b"same message", oaep)
print("Same message, different ciphertexts:", c1 != c2)
print("Decrypted:", private_key.decrypt(c1, oaep))

pss = padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH)
document = b"Contract: pay 1,000 USD"
signature = private_key.sign(document, pss, hashes.SHA256())
public_key.verify(signature, document, pss, hashes.SHA256())
print("Signature valid for the original document")
try:
    public_key.verify(signature, b"Contract: pay 9,000 USD", pss, hashes.SHA256())
except InvalidSignature:
    print("Signature rejected for the altered document")

Same message, different ciphertexts: True
Decrypted: b'same message'
Signature valid for the original document
Signature rejected for the altered document


#### **Key Takeaways**

* Never use textbook RSA: deterministic encryption leaks equal messages and small inputs.
* Use at least 2048-bit keys (3072 for long-term use), `e = 65537`, OAEP for encryption, and PSS for signatures.
* In practice, RSA encrypts or exchanges a symmetric key; the data itself is encrypted with AES-GCM.